In [ ]:
#Curățarea setului de date
import pandas as pd
# Încarcă fișierul
df = pd.read_csv("dataset_4.csv")
# Elimină spațiile suplimentare din numele coloanelor
df.columns = df.columns.str.strip()
# Elimină rândurile care conțin cel puțin un NaN
df_clean = df.dropna()
# Salvează într-un nou fișier (sau suprascrie)
df_clean.to_csv("date_curat.csv", index=False)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# Încărcare date
df = pd.read_csv("date_curat.csv")
df['Data Timp'] = pd.to_datetime(df['Data Timp'])
df = df.sort_values("Data Timp")  # sortare temporală
df.head()

#Analaiza exploatorie EDA
numeric_cols = df.columns[1:]
df[numeric_cols].describe()

# Histograme
df[numeric_cols].hist(bins=30, figsize=(12, 8))
plt.suptitle("Distribuții ale parametrilor", fontsize=16)
plt.tight_layout()
plt.show()

# Matrice de corelație
plt.figure(figsize=(10, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Matricea de corelație")
plt.show()

# Verificăm existența valorilor lipsă
print(df[numeric_cols].isnull().sum())

#isolation forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numeric_cols])
iso = IsolationForest(contamination=0.004, random_state=42)
df["anomalie"] = iso.fit_predict(X_scaled)
# Vizualizare anomalii în turbiditate
plt.figure(figsize=(12, 4))
plt.plot(df["Data Timp"], df["Turbiditate (FNU)"], label="Turbiditate")
plt.scatter(df[df["anomalie"] == -1]["Data Timp"],
            df[df["anomalie"] == -1]["Turbiditate (FNU)"],
            color="red", label="Anomalie")
plt.legend()
plt.title("Detectare anomalii turbiditate")
plt.xlabel("Timp")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Vizualizare anomalii temperatura
plt.figure(figsize=(12, 4))
plt.plot(df["Data Timp"], df["Temperatura (°C)"], label="Temperatura (°C)")
plt.scatter(df[df["anomalie"] == -1]["Data Timp"],
            df[df["anomalie"] == -1]["Temperatura (°C)"],
            color="red", label="Anomalie")
plt.legend()
plt.title("Detectare anomalii temperatura")
plt.xlabel("Timp")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Vizualizare anomalii ORP (mV)
plt.figure(figsize=(12, 4))
plt.plot(df["Data Timp"], df["ORP (mV)"], label="ORP (mV)")
plt.scatter(df[df["anomalie"] == -1]["Data Timp"],
            df[df["anomalie"] == -1]["ORP (mV)"],
            color="red", label="Anomalie")
plt.legend()
plt.title("Detectare anomalii ORP (mV)")
plt.xlabel("Timp")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Vizualizare anomalii conductivitate (μS/cm)
plt.figure(figsize=(12, 4))
plt.plot(df["Data Timp"], df["Conductivitate (μS/cm)"], label="Conductivitate (μS/cm)")
plt.scatter(df[df["anomalie"] == -1]["Data Timp"],
            df[df["anomalie"] == -1]["Conductivitate (μS/cm)"],
            color="red", label="Anomalie")
plt.legend()
plt.title("Detectare anomalii conductivitate (μS/cm)")
plt.xlabel("Timp")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Vizualizare anomalii fDOM (QSU)
plt.figure(figsize=(12, 4))
plt.plot(df["Data Timp"], df["fDOM (QSU)"], label="fDOM (QSU)")
plt.scatter(df[df["anomalie"] == -1]["Data Timp"],
            df[df["anomalie"] == -1]["fDOM (QSU)"],
            color="red", label="Anomalie")
plt.legend()
plt.title("Detectare anomalii fDOM (QSU)")
plt.xlabel("Timp")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#Clustering 
kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(X_scaled)
# PCA pentru vizualizare 2D
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["cluster"], cmap="viridis")
plt.title("Clustering KMeans (PCA)")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.show()

#Predicții Random Forest
X = df[numeric_cols].drop("pH", axis=1)
y = df["pH"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred)
plt.xlabel("Valori reale pH")
plt.ylabel("Valori prezise pH")
plt.plot([min(y), max(y)], [min(y), max(y)], color='red', linestyle='--')
plt.grid(True)
plt.title("Predicție pH - Random Forest")
plt.show()

#predicție temporală LSTM
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
# Selectăm doar pH și îl normalizăm
from sklearn.preprocessing import MinMaxScaler
ph_data = df[["Turbiditate (FNU)"]].copy()
scaler_lstm = MinMaxScaler()
ph_scaled = scaler_lstm.fit_transform(ph_data)
# Creăm secvențe pentru LSTM
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)
SEQ_LEN = 10
X_lstm, y_lstm = create_sequences(ph_scaled, SEQ_LEN)
# Împărțim setul
split = int(0.8 * len(X_lstm))
X_train_lstm, X_test_lstm = X_lstm[:split], X_lstm[split:]
y_train_lstm, y_test_lstm = y_lstm[:split], y_lstm[split:]
# Model LSTM
model_lstm = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, 1)),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_train_lstm, y_train_lstm, epochs=50, batch_size=8, verbose=0)
# Predicții
y_pred_lstm = model_lstm.predict(X_test_lstm)
y_pred_lstm_rescaled = scaler_lstm.inverse_transform(y_pred_lstm)
y_test_lstm_rescaled = scaler_lstm.inverse_transform(y_test_lstm)
# Plot
plt.figure(figsize=(10, 4))
plt.plot(y_test_lstm_rescaled, label="Real")
plt.plot(y_pred_lstm_rescaled, label="Prevăzut")
plt.legend()
plt.title("Predicție pH cu LSTM")
plt.show()

#Predciție ARIMA
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")
# Încarcă datele
df = pd.read_csv("date_curat.csv")
# Conversie datetime și setare index
df["Data Timp"] = pd.to_datetime(df["Data Timp"], errors="coerce")
df = df.dropna(subset=["Data Timp"])  # elimină datetimes invalide
df = df.set_index("Data Timp")
# Resample la 5 minute și interpolare dacă lipsesc unele valori
df = df.resample("5min").mean().interpolate()
# Forțăm frecvența în index
df.index.freq = pd.infer_freq(df.index)
if df.index.freq is None:
    df.index.freq = "5min"  # forțăm dacă nu detectează corect
# Serie target pentru ARIMA (ex: Turbiditate)
serie = df["Turbiditate (FNU)"]
# ARIMA model
model = ARIMA(serie, order=(2, 1, 2))  # p, d, q — pot fi ajustați
model_fit = model.fit()
# Predicție pe 20 pași în viitor
forecast = model_fit.forecast(steps=20)
# Plot rezultate
plt.figure(figsize=(12, 4))
plt.plot(serie[-100:], label="Istoric")
plt.plot(forecast, label="Forecast", color="red")
plt.title("Predicție ARIMA - Turbiditate")
plt.xlabel("Timp")
plt.ylabel("FNU")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


import pandas as pd
# Încarcă datele originale
df = pd.read_csv("date_curat.csv")
# Conversie Data Timp în datetime și sortare
df["Data Timp"] = pd.to_datetime(df["Data Timp"], errors="coerce")
df = df.dropna(subset=["Data Timp"])  # elimină rânduri cu datetime invalid
df = df.sort_values("Data Timp").reset_index(drop=True)
# Calculează indexul de split la 99%
split_idx = int(len(df) * 0.99)
# Împarte în două DataFrame-uri
df_99 = df.iloc[:split_idx]
df_last_1 = df.iloc[split_idx:]
# Salvează în fișiere CSV
df_99.to_csv("date_99.csv", index=False)
df_last_1.to_csv("date_last_1.csv", index=False)
# Setare Data Timp ca index datetime în df_99 și forțare frecvență
df_99 = df_99.set_index("Data Timp")
df_99 = df_99.asfreq(pd.infer_freq(df_99.index))
if df_99.index.freq is None:
    df_99.index.freq = "5min"  # forțăm frecvența la 5 minute dacă nu este detectată corect
print("Datele au fost împărțite și salvate cu succes.")

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

# Încarcă datele
df = pd.read_csv("date_99.csv")
# Conversie coloana "Data Timp" în datetime și setare ca index
df["Data Timp"] = pd.to_datetime(df["Data Timp"], errors="coerce")
df = df.dropna(subset=["Data Timp"])  # elimină datetime invalide
df = df.set_index("Data Timp")
# Sortează după index pentru siguranță
df = df.sort_index()
# Resample la frecvență fixă 5 minute și interpolare temporală pentru completarea lacunelor
df = df.resample("5min").mean().interpolate(method="time")
print(f"Lungime serie după resample și interpolare: {len(df)}")
print(f"Număr NaN după interpolare: {df['Turbiditate (FNU)'].isna().sum()}")

# Alege seria țintă
serie = df["Turbiditate (FNU)"]
# Construiește modelul ARIMA (parametrii p,d,q pot fi ajustați)
model = ARIMA(serie, order=(2, 1, 2))
model_fit = model.fit()
# Afișează sumarul modelului
print(model_fit.summary())

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# Încarcă ultimele 1% din date
df_test = pd.read_csv("date_last_1.csv")
# Setare index datetime
df_test["Data Timp"] = pd.to_datetime(df_test["Data Timp"], errors="coerce")
df_test = df_test.dropna(subset=["Data Timp"])
df_test = df_test.sort_values("Data Timp").reset_index(drop=True)
df_test = df_test.set_index("Data Timp")
# Forțăm frecvența (de ex 5min)
df_test = df_test.asfreq(pd.infer_freq(df_test.index))
if df_test.index.freq is None:
    df_test.index.freq = "5min"
# Facem predictia pentru lungimea testului
steps = len(df_test)
forecast = model_fit.forecast(steps=steps)
fig, ax = plt.subplots(figsize=(12,5))  # Plot clar
ax.plot(df_test.index, df_test["Turbiditate (FNU)"], label="Real")
ax.plot(df_test.index, forecast, label="Predicție ARIMA", color="red")
# Setăm formatul datei pe axa X
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
plt.xticks(rotation=45, ha='right')
plt.title("Comparație ARIMA - Real pentru ultimele 1% din date")
plt.xlabel("Data Timp")
plt.ylabel("Turbiditate (FNU)")
plt.grid(True)
plt.tight_layout()
plt.show()
 
